# AI-Assisted Model Reaction Mapping

Compare published model reactions against ModelSEED reactions using AI-driven
equivalence evaluation.

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Cell 1: Evaluate reaction equivalence between published and ModelSEED reactions
pubmod = MSModelUtil.from_cobrapy("models/PublishedModel.XML")
rxndata = session.cache.load("reactiondata")
rxn_mapping = {}
for rxn in pubmod.model.reactions:
    if rxn.id in rxndata:
        rxn_mapping.setdefault(rxn.id, {})
        if rxndata[rxn.id]["MSID"] is not None:
            other_rxn = _legacy.get_reaction_by_id(rxndata[rxn.id]["MSID"])
            if other_rxn is not None:
                rxn_mapping[rxn.id][rxndata[rxn.id]["MSID"]] = _legacy.evaluate_reaction_equivalence(
                    rxn, other_rxn, rxndata[rxn.id]["Match evidence"]
                )
        for index, other_hit in enumerate(rxndata[rxn.id]["Other matches"]):
            array = other_hit.split(":")
            other_rxn = _legacy.reaction_id_to_msid(array[0] + ":" + array[1])
            if other_rxn is not None:
                other_rxn = _legacy.get_reaction_by_id(other_rxn)
                rxn_mapping[rxn.id][rxndata[rxn.id]["MSID"]] = _legacy.evaluate_reaction_equivalence(
                    rxn, other_rxn, ":".join(array[2:])
                )
    session.cache.save("rxn_mapping", rxn_mapping)

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Cell 2: Load cached AI curation results
cache = _legacy._load_cached_curation("ReactionEquivalence")
print(len(cache))

In [ ]:
%run util.py

# Cell 3: Verify published model structure
pubmod = MSModelUtil.from_cobrapy("models/PublishedModel.XML")
print(len(pubmod.model.reactions))